# Reconstructed 06_v2 rolling Random Forest training and evaluation

**Status:** reconstructed from the archived `06_v2` run summary, archived fitted model parameters, feature-set/split manifests, and the earlier Random Forest notebook. This is intended to replace the missing source notebook **only after it has been executed in the frozen environment and validated against all archived outputs**.

The original run summary records the missing source notebook name as `06_v2_rolling_riskmap_evaluation_previous_week_weather_final`.


In [ ]:
from pathlib import Path
import json, sys, platform, warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

BASE_DIR = Path("/content/drive/MyDrive/avian_influenza_project")
PROC_DIR = BASE_DIR / "processed"
RESULT_DIR = PROC_DIR / "model_outputs_riskmap_eval"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

PANEL_PATH = PROC_DIR / "hpai_weekly_grid_panel_with_previous_week_weather_model_ready.parquet"
TARGET = "outbreak_binary"

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("platform:", platform.platform())
print("PANEL_PATH:", PANEL_PATH)


In [ ]:
panel = pd.read_parquet(PANEL_PATH)
panel["week_start"] = pd.to_datetime(panel["week_start"]).dt.normalize()

assert len(panel) == 1_641_809
assert panel["grid_id"].nunique() == 5_491
assert panel.duplicated(["grid_id", "week_start"]).sum() == 0
assert TARGET in panel.columns

BASE = [
    "grid_lat", "grid_lon", "weekofyear", "month", "sin_week", "cos_week",
    "temp_mean_c_lag1w", "temp_min_c_lag1w", "temp_max_c_lag1w",
]
SAME_GRID = ["lag_outbreak_1w", "lag_outbreak_2w", "lag_outbreak_4w", "lag_outbreak_8w"]
NEIGHBOR = [
    "neighbor_outbreak_count_past_1w", "neighbor_outbreak_count_past_2w",
    "neighbor_outbreak_count_past_4w", "neighbor_outbreak_count_past_8w",
]

MODEL_SPECS = {
    "geo_season_previous_week_weather": ("Geography–season–previous-week weather", BASE),
    "geo_season_previous_week_weather_same_grid_lags": ("Baseline + same-grid outbreak lags", BASE + SAME_GRID),
    "geo_season_previous_week_weather_neighbor_past": ("Baseline + neighboring past outbreaks", BASE + NEIGHBOR),
    "geo_season_previous_week_weather_lags_neighbor_past": ("Baseline + same-grid lags + neighboring past outbreaks", BASE + SAME_GRID + NEIGHBOR),
}

SPLITS = [
    ("test_fy2023", pd.Timestamp("2023-04-01"), pd.Timestamp("2024-04-01")),
    ("test_fy2024", pd.Timestamp("2024-04-01"), pd.Timestamp("2025-04-01")),
    ("test_fy2025", pd.Timestamp("2025-04-01"), pd.Timestamp("2026-04-01")),
]

for _, feats in MODEL_SPECS.values():
    missing=[x for x in feats if x not in panel.columns]
    if missing: raise KeyError(missing)

print("panel", panel.shape, "events", int(panel[TARGET].sum()))


In [ ]:
def add_weekly_rank_metrics(pred):
    parts=[]
    for week, g0 in pred.groupby("week_start", sort=True):
        g=g0.copy()
        n=len(g)
        g["n_grids_in_week"] = n
        g["risk_rank"] = g["pred_proba"].rank(method="average", ascending=False)
        g["risk_percentile"] = 1.0 - (g["risk_rank"] - 1.0) / (n - 1.0)
        for k in [0.01,0.05,0.10,0.20]:
            cutoff=int(np.ceil(n*k))
            g[f"top{int(k*100)}"] = g["risk_rank"] <= cutoff
        parts.append(g)
    return pd.concat(parts, ignore_index=True)

summary_rows=[]
event_parts=[]
feature_importance=[]

for split_name, test_start, test_end in SPLITS:
    train = panel.loc[panel.week_start < test_start].copy()
    test = panel.loc[(panel.week_start >= test_start) & (panel.week_start < test_end)].copy()
    print(split_name, "train", len(train), int(train[TARGET].sum()), "test", len(test), int(test[TARGET].sum()))

    for model_name,(display_label,features) in MODEL_SPECS.items():
        med=train[features].median(numeric_only=True)
        X_train=train[features].fillna(med)
        X_test=test[features].fillna(med)
        y_train=train[TARGET].astype(int)
        y_test=test[TARGET].astype(int)

        rf=RandomForestClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )
        rf.fit(X_train,y_train)
        score=rf.predict_proba(X_test)[:,1]

        keep=["grid_id","week_start",TARGET,"grid_lat","grid_lon"]
        if "outbreak_count" in test.columns: keep.append("outbreak_count")
        pred=test[keep].copy()
        pred["pred_proba"]=score
        pred["split_name"]=split_name
        pred["model_name"]=model_name
        pred["display_label"]=display_label
        pred=add_weekly_rank_metrics(pred)

        pred_path=RESULT_DIR / f"06_v2_rolling_{split_name}_{model_name}_predictions.parquet"
        model_path=RESULT_DIR / f"06_v2_rolling_{split_name}_{model_name}.joblib"
        pred.to_parquet(pred_path,index=False)
        joblib.dump(rf,model_path)

        ev=pred.loc[pred[TARGET].eq(1)].copy()
        event_parts.append(ev)
        base=float(y_test.mean())
        row={
            "split_name":split_name,"model_name":model_name,"display_label":display_label,
            "test_start":test_start.date().isoformat(),"test_end_exclusive":test_end.date().isoformat(),
            "test_rows":len(test),"n_events":int(y_test.sum()),"baseline_event_rate":base,
            "roc_auc":roc_auc_score(y_test,score),"pr_auc":average_precision_score(y_test,score),
            "mean_event_percentile":ev.risk_percentile.mean(),"median_event_percentile":ev.risk_percentile.median(),
            "mean_event_rank":ev.risk_rank.mean(),"median_event_rank":ev.risk_rank.median(),
            "pr_auc_over_baseline":average_precision_score(y_test,score)/base,
            "n_features":len(features),
        }
        for p in [1,5,10,20]:
            row[f"top{p}_events"]=int(ev[f"top{p}"].sum())
            row[f"top{p}_capture_rate"]=float(ev[f"top{p}"].mean())
        summary_rows.append(row)
        for f,imp in zip(features,rf.feature_importances_):
            feature_importance.append({"split_name":split_name,"model_name":model_name,"display_label":display_label,"feature":f,"importance":float(imp)})

summary_by_split=pd.DataFrame(summary_rows)
event_cases=pd.concat(event_parts,ignore_index=True)
feature_importance=pd.DataFrame(feature_importance)

summary_by_split.to_csv(RESULT_DIR/"06_v2_rolling_summary_by_split_previous_week_weather_RECONSTRUCTED.csv",index=False,encoding="utf-8-sig")
event_cases.to_csv(RESULT_DIR/"06_v2_rolling_event_cases_previous_week_weather_RECONSTRUCTED.csv",index=False,encoding="utf-8-sig")
feature_importance.to_csv(RESULT_DIR/"06_v2_rolling_feature_importance_previous_week_weather_RECONSTRUCTED.csv",index=False,encoding="utf-8-sig")


In [ ]:
pooled=[]
for model_name,(display_label,features) in MODEL_SPECS.items():
    g=event_cases.loc[event_cases.model_name.eq(model_name)].copy()
    row={
        "model_name":model_name,"display_label":display_label,"n_splits":3,"total_events":len(g),
        "mean_event_percentile":g.risk_percentile.mean(),"median_event_percentile":g.risk_percentile.median(),
        "mean_event_rank":g.risk_rank.mean(),"median_event_rank":g.risk_rank.median(),
    }
    for p in [1,5,10,20]:
        row[f"top{p}_events"]=int(g[f"top{p}"].sum())
        row[f"top{p}_capture_rate"]=float(g[f"top{p}"].mean())
    pooled.append(row)
pooled=pd.DataFrame(pooled)
pooled.to_csv(RESULT_DIR/"06_v2_rolling_pooled_summary_previous_week_weather_RECONSTRUCTED.csv",index=False,encoding="utf-8-sig")
display(pooled)


In [ ]:
# Validation against archived reported output. Adjust ARCHIVE path if running outside the public repository.
archive_candidates=[
    Path.cwd()/"reported_outputs"/"06_v2_rolling_pooled_summary_previous_week_weather.csv",
    Path.cwd().parent.parent/"reported_outputs"/"06_v2_rolling_pooled_summary_previous_week_weather.csv",
]
archive=next((p for p in archive_candidates if p.exists()),None)
if archive is None:
    print("Archived reference CSV not found; training finished but exact validation was skipped.")
else:
    ref=pd.read_csv(archive)
    cols=["total_events","mean_event_percentile","median_event_percentile","mean_event_rank","median_event_rank",
          "top1_events","top1_capture_rate","top5_events","top5_capture_rate","top10_events","top10_capture_rate","top20_events","top20_capture_rate"]
    a=pooled.set_index("model_name").sort_index()
    b=ref.set_index("model_name").sort_index()
    assert a.index.equals(b.index)
    failures=[]
    for c in cols:
        av=a[c].astype(float).to_numpy(); bv=b[c].astype(float).to_numpy()
        if not np.allclose(av,bv,rtol=0,atol=1e-12): failures.append(c)
    if failures:
        raise AssertionError(f"Reconstructed training did not exactly reproduce archived metrics: {failures}")
    print("SUCCESS: reconstructed 06_v2 training exactly reproduces archived pooled metrics.")
